# 05 Router-focused MoE x DQA Ten Loops

This notebook continues the MoE direction after `04_ten_research_loops`.
The previous result showed that day experts and neck/head day residuals
beat the single DQA aggregate.  This run asks a more router-specific
question:

- Can virtual routing explain the remaining gap?
- Can low-anchor expert mixing keep the best expert performance?
- Do BN statistics matter for expert behavior?
- Is top-k expert weighting better than uniform expert averaging?

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

import pandas as pd

cwd = Path.cwd().resolve()
if cwd.name == "notebooks" and cwd.parent.name == "moe":
    MOE_ROOT = cwd.parent
elif (cwd / "dynamic_quality_aware_classwise_aggregation").exists():
    MOE_ROOT = cwd / "dynamic_quality_aware_classwise_aggregation" / "scene_daynight_dqa" / "moe"
else:
    MOE_ROOT = cwd

SCENE_ROOT = MOE_ROOT.parent
WORKSPACE = MOE_ROOT / "output" / "05_router_ten_loops"
SOURCE_WORKSPACE = SCENE_ROOT / "output" / "02_head_to_full_long_dqa"
PREV_LOOP_WORKSPACE = MOE_ROOT / "output" / "04_ten_research_loops"
RUNNER = MOE_ROOT / "scripts" / "run_moe_05_router_ten_loops.py"

print("MOE_ROOT", MOE_ROOT)
print("SOURCE_WORKSPACE", SOURCE_WORKSPACE)
print("PREV_LOOP_WORKSPACE", PREV_LOOP_WORKSPACE)
print("WORKSPACE", WORKSPACE)
print("RUNNER", RUNNER)

## Setup / Sanity Check

In [ ]:
spec = importlib.util.spec_from_file_location("run_moe_05_router_ten_loops", RUNNER)
runner = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = runner
spec.loader.exec_module(runner)

args = runner.parse_args([
    "--workspace-root", str(WORKSPACE),
    "--source-workspace", str(SOURCE_WORKSPACE),
    "--prev-loop-workspace", str(PREV_LOOP_WORKSPACE),
    "--setup-only",
])
runner.run(args)

## Execute Ten Router Loops

In [ ]:
cmd = [
    sys.executable,
    str(RUNNER),
    "--workspace-root", str(WORKSPACE),
    "--source-workspace", str(SOURCE_WORKSPACE),
    "--prev-loop-workspace", str(PREV_LOOP_WORKSPACE),
    "--client-limit", "1500",
    "--evaluate",
    "--classwise",
    "--no-eval-plots",
    "--notify",
]
print(" ".join(cmd))
subprocess.run(cmd, cwd=MOE_ROOT, check=True)

## Results

In [ ]:
metrics_path = WORKSPACE / "stats" / "05_router_ten_loop_metrics.csv"
log_path = WORKSPACE / "stats" / "05_router_ten_loop_log.csv"

metrics = pd.read_csv(metrics_path)
display(
    metrics.sort_values("map50_95", ascending=False)[
        [
            "loop_id",
            "checkpoint_label",
            "map50",
            "map50_95",
            "gain_vs_warmup_map50_95",
            "night_avg_map50_95",
            "worst_split",
            "worst_split_map50_95",
            "variant",
        ]
    ].head(40)
)

loop_log = pd.read_csv(log_path)
display(loop_log)

## Markdown Report

In [ ]:
report = WORKSPACE / "05_router_ten_loop_report.md"
print(report)
print(report.read_text(encoding="utf-8")[:5000])